In [4]:
spark.table("fact_sales").printSchema()
spark.sql("SELECT MIN(order_date), MAX(order_date), COUNT(*) FROM fact_sales").show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 6, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- line_revenue: double (nullable = true)

+---------------+---------------+--------+
|min(order_date)|max(order_date)|count(1)|
+---------------+---------------+--------+
|     2024-03-07|     2026-03-07|24000000|
+---------------+---------------+--------+



In [5]:
spark.sql("DESC fact_sales").show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 7, Finished, Available, Finished, False)

+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|    order_id|   bigint|   NULL|
| customer_id|   bigint|   NULL|
|  product_id|   bigint|   NULL|
|  order_date|     date|   NULL|
|    quantity|   bigint|   NULL|
|line_revenue|   double|   NULL|
+------------+---------+-------+



###### **<mark>Step 1: Create an incremental target demo table</mark>**

In [6]:
spark.sql("DROP TABLE IF EXISTS fact_sales_incremental_demo")

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 8, Finished, Available, Finished, False)

DataFrame[]

In [7]:
spark.table("fact_sales").write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales_incremental_demo")

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 9, Finished, Available, Finished, False)

###### **<mark>Step 2: Capture current watermark</mark>**

In [8]:
watermark_row = spark.sql("""
SELECT MAX(order_date) AS last_loaded_date
FROM fact_sales_incremental_demo
""").collect()[0]

last_loaded_date = watermark_row["last_loaded_date"]

print("Current watermark:", last_loaded_date)

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 10, Finished, Available, Finished, False)

Current watermark: 2026-03-07


###### **<mark>Step 3: Create a simulated incoming batch</mark>**

In [9]:
from pyspark.sql.types import StructType, StructField, LongType, DateType, DoubleType
from datetime import date

incoming_orders_data = [
    (999000001, 101, 2001, date(2026, 3, 5),  2, 150.00),  # old, ignore
    (999000002, 102, 2002, date(2026, 3, 7),  1,  80.00),  # same watermark day, ignore if using >
    (999000003, 103, 2003, date(2026, 3, 8),  3, 250.00),  # new, load
    (999000004, 104, 2004, date(2026, 3, 9),  5, 420.00)   # new, load
]

incoming_orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("product_id", LongType(), True),
    StructField("order_date", DateType(), True),
    StructField("quantity", LongType(), True),
    StructField("line_revenue", DoubleType(), True)
])

incoming_orders_df = spark.createDataFrame(
    incoming_orders_data,
    schema=incoming_orders_schema
)

incoming_orders_df.show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 11, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+
| order_id|customer_id|product_id|order_date|quantity|line_revenue|
+---------+-----------+----------+----------+--------+------------+
|999000001|        101|      2001|2026-03-05|       2|       150.0|
|999000002|        102|      2002|2026-03-07|       1|        80.0|
|999000003|        103|      2003|2026-03-08|       3|       250.0|
|999000004|        104|      2004|2026-03-09|       5|       420.0|
+---------+-----------+----------+----------+--------+------------+



###### **<mark>Step 4: Filter incrementally using watermark</mark>**`

**We will use:**

<mark>order_date > last_loaded_date</mark>

**That means:**

<mark>rows on 2026-03-07 are not loaded again
only truly newer records are loaded</mark>

In [10]:
incremental_orders_df = incoming_orders_df.filter(
    incoming_orders_df["order_date"] > last_loaded_date
)

incremental_orders_df.show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 12, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+
| order_id|customer_id|product_id|order_date|quantity|line_revenue|
+---------+-----------+----------+----------+--------+------------+
|999000003|        103|      2003|2026-03-08|       3|       250.0|
|999000004|        104|      2004|2026-03-09|       5|       420.0|
+---------+-----------+----------+----------+--------+------------+



###### **<mark>Step 5: Append only incremental rows</mark>**

In [11]:
incremental_orders_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("fact_sales_incremental_demo")

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 13, Finished, Available, Finished, False)

###### <mark>**Step 6: Validate incremental result**</mark>
**Count check**

In [12]:
spark.sql("""
SELECT COUNT(*) AS total_rows
FROM fact_sales_incremental_demo
""").show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 14, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|  24000002|
+----------+



###### <mark>**New watermark check**</mark>

In [13]:
spark.sql("""
SELECT MAX(order_date) AS new_watermark
FROM fact_sales_incremental_demo
""").show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 15, Finished, Available, Finished, False)

+-------------+
|new_watermark|
+-------------+
|   2026-03-09|
+-------------+



<mark>**New rows check**</mark>

In [14]:
spark.sql("""
SELECT *
FROM fact_sales_incremental_demo
WHERE order_id IN (999000001, 999000002, 999000003, 999000004)
ORDER BY order_id
""").show()

StatementMeta(, 8912e3cc-8757-444b-a8dd-bb3d1c9f347a, 16, Finished, Available, Finished, False)

+---------+-----------+----------+----------+--------+------------+
| order_id|customer_id|product_id|order_date|quantity|line_revenue|
+---------+-----------+----------+----------+--------+------------+
|999000003|        103|      2003|2026-03-08|       3|       250.0|
|999000004|        104|      2004|2026-03-09|       5|       420.0|
+---------+-----------+----------+----------+--------+------------+

